In [46]:
import pandas as pd
import numpy as np
import random
import sklearn.manifold as manifold
import colorsys

import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots

In [47]:
# Read acceleration data
acceleration_data = pd.read_csv("../output/handshape_acceleration.csv")

acceleration_data.head()

,frame,time,acceleration
0,0,0.000000,0.377796
1,1,0.016667,0.403649
2,2,0.033333,0.434516
3,3,0.050000,0.460212
4,4,0.066667,0.476669


In [48]:
# Read segment data
segment_data = pd.read_csv("../output/handshape_segments.csv")

segment_data.head()

,start,end,average_pose
0,0,91,RightHandMiddle1:0.956914 0.0252167 0.289176 -...
1,103,131,RightHandMiddle1:0.810752 -0.0478933 0.582412 ...
2,135,176,RightHandMiddle1:0.812837 -0.0546305 0.578618 ...
3,193,201,RightHandMiddle1:0.845586 -0.0501044 0.530552 ...
4,207,219,RightHandMiddle1:0.838272 -0.0543248 0.541403 ...


In [49]:
# # Add hand_index data to acceleration_data (so add an index based on the hand and its)
# acceleration_data['hand_index'] = acceleration_data.groupby('hand').cumcount()

# acceleration_data.head()

In [50]:
fig = go.Figure()

fig.add_trace(
    go.Scatter(
        x=acceleration_data["frame"],
        y=acceleration_data["acceleration"],
        mode="lines",
        line=dict(color="#1f77b4", width=1.5),
        name="Base",
        legendgroup="Base",
    )
)

colours = px.colors.qualitative.Plotly

for index, row in segment_data.iterrows():
    current_color = colours[index % len(colours)]

    segment_slice = acceleration_data[
        (acceleration_data["frame"] >= row["start"])
        & (acceleration_data["frame"] <= row["end"])
    ]

    if not segment_slice.empty:
        fig.add_trace(
            go.Scatter(
                x=segment_slice["frame"],
                y=segment_slice["acceleration"],
                mode="lines",
                line=dict(color=current_color, width=4),
                name=f"Seg {index}",
                legendgroup=f"Segment {index}",
                showlegend=True,
            )
        )

fig.update_layout(
    title_text="Hand Acceleration Data",
    height=600,
    showlegend=True,
    xaxis_title="Time (s)",
    yaxis_title="Acceleration (rad/s²)",
)

fig.show()

Note:

.\fbx\M20260119_1078_260713_1.fbx

Segment 4 wordt als 1 segment gezien.
De piek is boven de threshold, dus wordt gezien. Het dal wordt niet gezien, omdat het onder de threshold is.
Het zou dus mogelijk zijn om dit als 2 segmenten te kunnen detecteren.
Er is bewust gekozen om dat niet aan te passen, omdat deze situatie aangeeft dat het verschil tussen deze poses heel subtiel zal zijn (in dit geval is het verschil {pose} en {relaxed_pose}).
Dit subtiele verschil geeft voor beide delen van dit segment dezelfde pose, dus hoeven deze niet apart gedetecteerd te worden.